## Lab 6: IIR filters for harmonic envelopes

In this lab we use an IIR filter to isolate every single harmonic of a sound, estimate its envelope, and then use that envelope to resynthesize a sound aiming to keep the time varying harmonic structure of the original sound.

The key idea from the IIR theory summary is that an IIR filter uses both the current input and previous outputs. A general recursive filter can be written as

$y[n] = \sum_{k=0}^{M} b_k x[n-k] + \sum_{k=1}^{N} a_k y[n-k]$.

In the simple envelope detector used here, the recursion becomes

$y[n] = (1-r) |x[n]| + r y[n-1]$, with $0<r<1$.

Because the output depends on its previous value, the envelope changes smoothly and responds more slowly than a plain instantaneous estimate. This is exactly the behaviour we want when we try to track the amplitude of one harmonic over time.

The next cell includes the basic functions needed for the whole lab.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from IPython.display import Audio, display
from util import plot_signals

def synthesize(f0, phi, Ak, t):
    """Build a sum of harmonics with the given amplitudes and phase."""
    y = np.zeros_like(t, dtype=float)
    for k in range(1, len(Ak) + 1):
        y += Ak[k - 1] * np.cos(2 * np.pi * k * f0 * t + k * phi - (k - 1) * np.pi / 2)
    return y

def iir_envelope(x, r=0.98):
    """Compute a smooth amplitude envelope with a first-order recursive IIR filter."""
    x = np.asarray(x, dtype=float)
    y = np.zeros_like(x)
    for n in range(1, len(x)):
        y[n] = (1 - r) * np.abs(x[n]) + r * y[n - 1]
    return y

def bandpass_harmonic(x, f_center, fs, bw=25, order=2):
    """Filter a signal around a chosen harmonic frequency with a band-pass filter."""
    if f_center <= 0:
        return np.zeros_like(x)
    low = max(1e-6, (f_center - bw / 2) / (fs / 2))
    high = min(1.0 - 1e-6, (f_center + bw / 2) / (fs / 2))
    b, a = signal.butter(order, [low, high], btype="bandpass")
    return signal.lfilter(b, a, x)

def extract_harmonic_envelopes(x, f0, fs, n_harmonics=5, bw=25, r=0.98):
    """Extract one envelope for each harmonic by filtering and smoothing it."""
    envelopes = []
    for k in range(1, n_harmonics + 1):
        component = bandpass_harmonic(x, k * f0, fs, bw=bw)
        envelopes.append(iir_envelope(component, r=r))
    return envelopes



### 1. Isolate one harmonic with an IIR band-pass filter

The first step is to isolate a single harmonic from the compound signal. We can do that with a band-pass filter centered at the harmonic frequency. The band-pass filter keeps frequencies near the chosen center frequency and attenuates the others, so it emphasizes the partial we want to study.

From the IIR point of view, the filter is still a recursive system: it uses previous output samples as well as the current input. In the frequency domain, this means the response is high around a chosen center frequency $f_c$ and lower away from it. In practice we choose a bandwidth that is wide enough to pass the harmonic but narrow enough to avoid picking up neighboring components.

Once the harmonic component is isolated, we can estimate its envelope by applying the recursive smoothing rule from the previous section. The idea is that the band-pass filter gives us the harmonic component, and the IIR smoothing step tells us how its strength evolves over time.

Next cell show how the envelope extraction of harmonics works on a synthesized reference sound.

In [2]:
# Synthesize a harmonic sound at 440 Hz and extract the first n harmonic envelopes.

# Sampling rate and time axis
fs = 22050
t = np.arange(0, 0.5, 1 / fs)

# Harmonic synthesis parameters
phi = 0.0
weights = [0.8, 0.5, 0.3]
f0_1 = 220.0
n_harmonics = 3  # change this value to analyze more or fewer harmonics
synthetic_signal = synthesize(f0_1, phi, weights, t)
synthetic_signal = synthetic_signal / np.max(np.abs(synthetic_signal))

synthetic_envelopes = extract_harmonic_envelopes(
    synthetic_signal, f0_1, fs, n_harmonics=n_harmonics, bw=25, r=0.98
)
synthetic_envelope_array = np.array(synthetic_envelopes)

# Prepare the list of signals to plot
signals_to_plot = [synthetic_signal]
for env in synthetic_envelopes:
    signals_to_plot.append(env * 0.7)

# Prepare the list of names for the legend
signal_names = ["harmonic signal"]
for k in range(1, n_harmonics + 1):
    signal_names.append(f"harmonic {k} envelope")

# Calculate the end time for the plot view from the sample count (4000)
t_end_s = 4000 / fs

# Use plot_signals to create the plot.
# Note: The title will not be displayed as the plot_signals function does not support it.
plot_signals(signals_to_plot, sr=fs, t_end=t_end_s, name=signal_names)



#### 1.1 Analyze the envelopes of the reference sound

Now we turn to a real sound. The goal here is to do it on your reference sound, isolating all frequency regions, and creating the envelops using IIR filters. You should use the values from previous labs to decide which parameters make the envelope look most meaningful for the sound you chose.

Adapt the following code to work with your sound and set the parameters that can best extract as many harmonic envelopes as possible.

In [7]:
from pathlib import Path
from IPython.display import Audio, display
from util import load_audio, plot_signals

# Clone your own repository and point reference_path to your own audio file.
# \!git clone https://github.com/your-username/sis1_name.git
# reference_path = Path('./sis1_name/your_file.wav')
reference_path = Path('my_pluck.wav')
reference_audio, fs = load_audio(reference_path)

x = reference_audio / np.max(np.abs(reference_audio))

# Specify the fundamental frequency, how many harmonics you want to study, and how many seconds to display.
f0_2 = 102.5  # replace with the value you want to analyze
n_harmonics = 5  # change this value to analyze more or fewer harmonics
plot_duration = 1.0  # choose how many seconds to display (default: 1 s)

# Generate the list of frequencies to analyze.
frequencies = [f0_2 * k for k in range(1, n_harmonics + 1)]

# Apply the band-pass filter and estimate the envelope for each chosen frequency.
n_samples = int(min(plot_duration, len(x) / fs) * fs)
t = np.arange(n_samples) / fs
reference_signal = x[:n_samples]

display(Audio(reference_signal, rate=fs))

# Extract the actual envelopes
ref_envelopes = extract_harmonic_envelopes(
    reference_signal, f0_2, fs, n_harmonics=n_harmonics, bw=25, r=0.98
)

# Prepare the list of signals to plot.
# We'll plot the main synthetic signal and each of its envelopes.
signals_to_plot = [reference_signal]
for env in ref_envelopes:
    signals_to_plot.append(env * 2.5)

# Prepare the list of names for the legend in the plot.
signal_names = ["harmonic signal"]
for k in range(1, n_harmonics + 1):
    signal_names.append(f"harmonic {k} envelope")

# We are plotting the first 4000 samples, so we calculate the end time in seconds.
# This assumes the 'sr' (sample rate) variable is defined.
t_end_s = 4000 / fs

# Use the plot_signals helper to display the graph.
# This single call replaces the entire matplotlib setup.
plot_signals(signals_to_plot, sr=fs, t_end=plot_duration, name=signal_names)



#### 1.2 Questions
To answer the following questions you might need to change the plot in order to better view specific aspects of the filtered harmonics and their envelopes. Specially you might want to make different zooms.

1. A part from the harmonic dependent amplitude scaling, are there particular time behaviours of the different harmonics? Explain and give some examples.
   - Yes, higher harmonics decay much faster over time due to natural string damping. In the plot, harmonics 3, 4, and 5 peak briefly right after the pluck around and fade out almost completely before t = 0.1s, whereas the fundamental frequency sustains its energy all the way out to around t = 0.4s.

2. How can you explain the smooth transition at the beginning of the sound/envelopes?
   - Even though the raw audio signal spikes almost instantaneously when the string is plucked, the envelopes curve upward smoothly, peaking later. This happen because both the bandpass filter and the envelope detector start at zero and depend heavily on previous output values (r = 0.98), so the filter builds up gradually over several samples instead of jumping straight to the peak.

4. Experiment with the smoothing parameter $r$ in the IIR envelope detector. How does it affect the result?
   - The smoothing parameter $r$ controls the trade-off between smoothness and response speed. A larger r eliminates the small oscillations visible on top of the orange curve, making it completely smooth, but increases filter inertia, causing a slower rise time and lagging behind the actual decay. A smaller r allows the envelope to react much faster to the sharp pluck attack, but leaves much larger high-frequency oscillations on the curve instead of a clean volume outline.

## 2. Resynthesize the signal from the harmonic envelopes

Once we have an envelope for each harmonic, we can rebuild the sound by multiplying each harmonic by its own slowly varying envelope. This is a simple model of amplitude modulation: the pitch stays the same, but the loudness of each partial changes over time.

In [8]:
# Example: reconstruct the synthesized signal from the envelopes extracted in section 1.
# This cell uses the synthetic signal and envelopes from the previous synthesis cell.

signal_to_resynthesize = synthetic_signal
signal_length = len(signal_to_resynthesize)
t_resynth = np.arange(signal_length) / fs

resynthesized = np.zeros_like(signal_to_resynthesize)
for harmonic_index, envelope in enumerate(synthetic_envelope_array, start=1):
    component = (
        weights[harmonic_index - 1]
        * envelope[:signal_length]
        * np.cos(
            2 * np.pi * harmonic_index * f0_1 * t_resynth
            + harmonic_index * phi
            - (harmonic_index - 1) * np.pi / 2
        )
    )
    resynthesized += component

resynthesized = resynthesized / np.max(np.abs(resynthesized))

display(Audio(signal_to_resynthesize, rate=fs))
display(Audio(resynthesized, rate=fs))
plot_signals(
    [signal_to_resynthesize, resynthesized],
    fs,
    t_start=0,
    t_end=min(0.4, signal_length / fs),
    name=['original', 'resynthesized'],
)


### 2.1 Resynthesize the reference sound

Now you should reconstruct your reference sound based on the analysis done in 1.2 

In [10]:
# Convert the list of envelopes from Section 1.1 into the expected array
reference_envelope_array = np.array(ref_envelopes)

# Rebuild the reference signal from the envelopes extracted in section 1.
# This cell uses the reference signal and envelopes from the analysis cell.

signal_to_resynthesize = reference_signal
signal_length = len(signal_to_resynthesize)
t_resynth = np.arange(signal_length) / fs

resynthesized = np.zeros_like(signal_to_resynthesize)
for harmonic_index, envelope in enumerate(reference_envelope_array, start=1):
    component = (
        envelope[:signal_length]
        * np.cos(
            2 * np.pi * harmonic_index * f0_2 * t_resynth
            + harmonic_index * phi
            - (harmonic_index - 1) * np.pi / 2
        )
    )
    resynthesized += component

resynthesized = resynthesized / np.max(np.abs(resynthesized))

display(Audio(signal_to_resynthesize, rate=fs))
display(Audio(resynthesized, rate=fs))
plot_signals(
    [signal_to_resynthesize, resynthesized],
    fs,
    t_start=0,
    t_end=min(0.5, signal_length / fs),
    name=["original", "resynthesized"],
)


### 2.2 Questions

1. What are the differences between original and reconstructed sound? explain the reason?
   - The reconstructed signal follows the overall pitch and decay of the original, but its waveform is much smoother, its attack swells in later, and it loses the sharp jagged details of the pluck. This happens because we only used the first 5 harmonics, missing all higher-frequency overtones and broadband pluck noise, and because the IIR bandpass and envelope filters introduce a slight delay and smooth out the rapid initial transient.

2. How could you improve the analysis/reconstruction process in order to make the synthesis sound more similar to the reference? explain why?
   - Firstly, you could extract and resynthesize more harmonics to bring back the high-frequency brightness. You could also use a slightly smaller smoothing parameter r so the attack transient isn't delayed. Or you could add a short burst of filtered noise to replicate the physical pluck of the string.

2. If the reference sound was an inharmonic sound, what would yo have to change in the analysis? explain?
   -  Instead of placing the bandpass filters at integer multiples of the fundamental frequency, you would need to detect the actual partial peaks directly from the signal's FFT and set the center frequencies of both the bandpass filters and the resynthesized cosines to those exact inharmonic frequencies.